# PyBullet Two-Object Tabletop Collision: Multi-View Video Capture

This notebook uses `phys_sim`, SI units, PyBullet rigid-body dynamics, and TinyRenderer to simulate two tabletop objects crashing into each other. The scene includes gravity, tabletop support, friction, restitution, rigid-body inertia, and contact resolution.

In [ ]:
import sys
from pathlib import Path

assert "phys_sim" in sys.executable, (
    f"Expected the phys_sim virtual environment, but got: {sys.executable}\n"
    "In Jupyter, select the kernel named 'phys_sim'."
)

import numpy as np
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
from IPython.display import Video, display

print(f"Python executable: {sys.executable}")
print(f"PyBullet data path: {pybullet_data.getDataPath()}")


## Scenario Parameters

The scene uses two measured blocks moving toward each other on a tabletop. Mass, dimensions, initial speeds, friction, and restitution are explicit so you can tune elastic or inelastic crashes.

In [ ]:
OUTPUT_DIR = Path("pybullet_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

GRAVITY = -9.80665
SIM_HZ = 360
TIME_STEP = 1.0 / SIM_HZ
VIDEO_FPS = 60
STEPS_PER_FRAME = SIM_HZ // VIDEO_FPS
DURATION_SEC = 2.6
N_STEPS = int(DURATION_SEC * SIM_HZ)

TABLE_LENGTH = 1.20
TABLE_WIDTH = 0.80
TABLE_THICKNESS = 0.06
TABLE_TOP_Z = 0.75
TABLE_CENTER_Z = TABLE_TOP_Z - TABLE_THICKNESS / 2.0
TABLE_RESTITUTION = 0.10
TABLE_FRICTION = 0.02

OBJECT_HALF_EXTENTS = [0.050, 0.050, 0.025]
OBJECT_A_MASS = 0.090
OBJECT_B_MASS = 0.060
OBJECT_A_START = [-0.36, 0.0, TABLE_TOP_Z + OBJECT_HALF_EXTENTS[2]]
OBJECT_B_START = [0.36, 0.0, TABLE_TOP_Z + OBJECT_HALF_EXTENTS[2]]
OBJECT_A_INITIAL_VELOCITY = [1.05, 0.0, 0.0]
OBJECT_B_INITIAL_VELOCITY = [-0.85, 0.0, 0.0]

OBJECT_FRICTION = 0.02
OBJECT_RESTITUTION = 0.80

IMG_WIDTH = 640
IMG_HEIGHT = 368

print(f"Writing outputs to: {OUTPUT_DIR.resolve()}")


In [ ]:
def connect_pybullet(deformable=False):
    if p.isConnected():
        p.disconnect()
    client = p.connect(p.DIRECT)
    if deformable:
        p.resetSimulation(p.RESET_USE_DEFORMABLE_WORLD, physicsClientId=client)
    else:
        p.resetSimulation(physicsClientId=client)
    p.setAdditionalSearchPath(pybullet_data.getDataPath(), physicsClientId=client)
    p.setGravity(0, 0, GRAVITY, physicsClientId=client)
    p.setTimeStep(TIME_STEP, physicsClientId=client)
    p.setPhysicsEngineParameter(
        fixedTimeStep=TIME_STEP,
        numSolverIterations=180,
        numSubSteps=2,
        restitutionVelocityThreshold=0.0,
        deterministicOverlappingPairs=1,
        physicsClientId=client,
    )
    return client


def create_table(client):
    p.loadURDF("plane.urdf", physicsClientId=client)
    collision = p.createCollisionShape(
        p.GEOM_BOX,
        halfExtents=[TABLE_LENGTH / 2, TABLE_WIDTH / 2, TABLE_THICKNESS / 2],
        physicsClientId=client,
    )
    visual = p.createVisualShape(
        p.GEOM_BOX,
        halfExtents=[TABLE_LENGTH / 2, TABLE_WIDTH / 2, TABLE_THICKNESS / 2],
        rgbaColor=[0.58, 0.40, 0.24, 1.0],
        physicsClientId=client,
    )
    table_id = p.createMultiBody(
        baseMass=0,
        baseCollisionShapeIndex=collision,
        baseVisualShapeIndex=visual,
        basePosition=[0, 0, TABLE_CENTER_Z],
        physicsClientId=client,
    )
    p.changeDynamics(
        table_id,
        -1,
        restitution=TABLE_RESTITUTION,
        lateralFriction=TABLE_FRICTION,
        spinningFriction=0.01,
        rollingFriction=0.01,
        physicsClientId=client,
    )
    return table_id

def camera_matrices(view_name, target):
    cameras = {
        "front": {"eye": [0.0, -1.65, 1.08], "up": [0, 0, 1], "fov": 50},
        "side": {"eye": [1.65, 0.0, 1.08], "up": [0, 0, 1], "fov": 50},
        "back": {"eye": [0.0, 1.65, 1.08], "up": [0, 0, 1], "fov": 50},
        "top": {"eye": [0.0, 0.0, 2.35], "up": [0, 1, 0], "fov": 44},
    }
    spec = cameras[view_name]
    view = p.computeViewMatrix(spec["eye"], target, spec["up"])
    proj = p.computeProjectionMatrixFOV(
        fov=spec["fov"],
        aspect=IMG_WIDTH / IMG_HEIGHT,
        nearVal=0.02,
        farVal=5.0,
    )
    return view, proj


CAMERA_NAMES = ["front", "side", "back", "top"]


def render_rgb(client, view_name, target):
    view, proj = camera_matrices(view_name, target)
    _, _, rgba, _, _ = p.getCameraImage(
        width=IMG_WIDTH,
        height=IMG_HEIGHT,
        viewMatrix=view,
        projectionMatrix=proj,
        renderer=p.ER_TINY_RENDERER,
        lightDirection=[-0.5, -0.4, -1.0],
        physicsClientId=client,
    )
    rgba = np.asarray(rgba, dtype=np.uint8).reshape(IMG_HEIGHT, IMG_WIDTH, 4)
    return rgba[:, :, :3]


def make_writers(prefix):
    paths = {name: OUTPUT_DIR / f"{prefix}_{name}.mp4" for name in CAMERA_NAMES}
    writers = {
        name: imageio.get_writer(path, fps=VIDEO_FPS, codec="libx264", quality=8, macro_block_size=16)
        for name, path in paths.items()
    }
    return paths, writers

def make_collision_object(client, name, mass, position, velocity, rgba):
    collision = p.createCollisionShape(p.GEOM_BOX, halfExtents=OBJECT_HALF_EXTENTS, physicsClientId=client)
    visual = p.createVisualShape(p.GEOM_BOX, halfExtents=OBJECT_HALF_EXTENTS, rgbaColor=rgba, physicsClientId=client)
    body = p.createMultiBody(mass, collision, visual, position, physicsClientId=client)
    p.changeDynamics(
        body,
        -1,
        lateralFriction=OBJECT_FRICTION,
        restitution=OBJECT_RESTITUTION,
        rollingFriction=0.001,
        spinningFriction=0.001,
        physicsClientId=client,
    )
    p.resetBaseVelocity(body, linearVelocity=velocity, physicsClientId=client)
    return name, body, mass


def create_collision_scene(client):
    table_id = create_table(client)
    bodies = [
        make_collision_object(
            client,
            "object_a",
            OBJECT_A_MASS,
            OBJECT_A_START,
            OBJECT_A_INITIAL_VELOCITY,
            [0.08, 0.30, 0.90, 1],
        ),
        make_collision_object(
            client,
            "object_b",
            OBJECT_B_MASS,
            OBJECT_B_START,
            OBJECT_B_INITIAL_VELOCITY,
            [0.88, 0.15, 0.12, 1],
        ),
    ]
    return table_id, bodies


## Run Simulation and Record Videos

The two object poses are sampled at the same ticks as the four cameras, so the CSV and videos are synchronized.

In [ ]:
client = connect_pybullet()
table_id, bodies = create_collision_scene(client)
video_paths, writers = make_writers("tabletop_collisions")
records = []

try:
    for step in range(N_STEPS + 1):
        t = step * TIME_STEP
        row = [t]
        total_ke = 0.0
        total_momentum_xy = np.zeros(2)
        for name, body_id, mass in bodies:
            pos, orn = p.getBasePositionAndOrientation(body_id, physicsClientId=client)
            lin_vel, ang_vel = p.getBaseVelocity(body_id, physicsClientId=client)
            row.extend([pos[0], pos[1], pos[2], lin_vel[0], lin_vel[1], lin_vel[2]])
            speed2 = np.dot(lin_vel, lin_vel)
            total_ke += 0.5 * mass * speed2
            total_momentum_xy += mass * np.asarray(lin_vel[:2])
        row.extend([total_ke, total_momentum_xy[0], total_momentum_xy[1]])
        records.append(row)

        if step % STEPS_PER_FRAME == 0:
            target = [0.0, 0.0, TABLE_TOP_Z + 0.12]
            for name, writer in writers.items():
                writer.append_data(render_rgb(client, name, target))

        p.stepSimulation(physicsClientId=client)
finally:
    for writer in writers.values():
        writer.close()
    p.disconnect(client)

records = np.asarray(records)
columns = ["time"]
for name, _, _ in bodies:
    columns.extend([f"{name}_x", f"{name}_y", f"{name}_z", f"{name}_vx", f"{name}_vy", f"{name}_vz"])
columns.extend(["total_translational_ke", "total_px", "total_py"])
trajectory_path = OUTPUT_DIR / "tabletop_collisions_trajectory.csv"
np.savetxt(trajectory_path, records, delimiter=",", header=",".join(columns), comments="")

print("Saved videos:")
for name, path in video_paths.items():
    print(f"  {name:>5}: {path}")
print(f"Saved trajectory: {trajectory_path}")


## Quick Physics Checks

In [ ]:
times = records[:, 0]
a_pos = records[:, 1:4]
a_vel = records[:, 4:7]
b_pos = records[:, 7:10]
b_vel = records[:, 10:13]
separation_xy = np.linalg.norm(a_pos[:, :2] - b_pos[:, :2], axis=1)
closest_idx = int(np.argmin(separation_xy))
pre_idx = max(closest_idx - int(0.05 * SIM_HZ), 0)
post_idx = min(closest_idx + int(0.08 * SIM_HZ), len(records) - 1)

initial_ke = records[0, -3]
final_ke = records[-1, -3]
initial_p = records[0, -2:]
final_p = records[-1, -2:]
relative_speed_before = a_vel[pre_idx, 0] - b_vel[pre_idx, 0]
relative_speed_after = b_vel[post_idx, 0] - a_vel[post_idx, 0]
measured_restitution = relative_speed_after / relative_speed_before if relative_speed_before != 0 else np.nan

print(f"Closest approach time:         {times[closest_idx]:.4f} s")
print(f"Closest center separation:     {separation_xy[closest_idx]:.4f} m")
print(f"Nominal contact separation:    {2 * OBJECT_HALF_EXTENTS[0]:.4f} m")
print(f"Relative approach speed:       {relative_speed_before:.4f} m/s")
print(f"Relative separation speed:     {relative_speed_after:.4f} m/s")
print(f"Measured restitution approx.:  {measured_restitution:.3f}")
print(f"Configured restitution:        {OBJECT_RESTITUTION:.3f}")
print(f"Initial translational KE:      {initial_ke:.4f} J")
print(f"Final translational KE:        {final_ke:.4f} J")
print(f"Initial xy momentum:           {np.round(initial_p, 4)} kg*m/s")
print(f"Final xy momentum:             {np.round(final_p, 4)} kg*m/s")
print("Small momentum changes are expected because tabletop friction is an external force.")


## Preview Videos

In [ ]:
for name in CAMERA_NAMES:
    print(name)
    display(Video(str(video_paths[name]), embed=True, html_attributes="controls loop"))
